# Sequence 1 : Bases de la theorie des graphes

Ce notebook introduit les concepts fondamentaux de la theorie des graphes necessaires pour aborder les boucles du bloc de Recherche Operationnelle. L'objectif est de construire une intuition solide sur les notions de graphe, de parcours et de plus court chemin, en les manipulant concretement en Python.

Le notebook est concu pour etre execute de haut en bas : plusieurs cellules reutilisent des variables definies dans les sections precedentes.

## 1. Vocabulaire et definitions

Un **graphe** est une structure mathematique composee de **sommets** (ou noeuds) et d'**aretes** (ou arcs) qui relient ces sommets entre eux.

Quelques definitions essentielles :
- **Graphe non oriente** : les aretes n'ont pas de direction (A--B est identique a B--A)
- **Graphe oriente (digraphe)** : les arcs ont une direction (A->B est different de B->A)
- **Graphe pondere** : chaque arete porte un poids (distance, cout, duree...)
- **Degre d'un sommet** : nombre d'aretes incidentes a ce sommet
- **Chemin** : suite de sommets relies par des aretes
- **Cycle** : chemin dont le sommet de depart et le sommet d'arrivee sont identiques
- **Graphe connexe** : il existe un chemin entre toute paire de sommets

Dans le contexte du projet ADEME, les villes sont des sommets et les routes sont des aretes ponderees par la duree du trajet.

## 2. Representations d'un graphe en Python

Il existe deux grandes facons de representer un graphe en memoire, chacune avec ses avantages et ses inconvenients.

### 2.1 Liste d'adjacence (dictionnaire)

Chaque sommet est associe a la liste de ses voisins. Cette representation est compacte pour les graphes creux (peu d'aretes par rapport au nombre de sommets).

In [ ]:
# Graphe non oriente et pondere : reseau de 5 villes
# Chaque cle est un sommet, chaque valeur est une liste de tuples (voisin, poids)
reseau = {
    "Paris":     [("Lyon", 460), ("Lille", 220), ("Nantes", 385)],
    "Lyon":      [("Paris", 460), ("Marseille", 315), ("Nantes", 690)],
    "Lille":     [("Paris", 220)],
    "Nantes":    [("Paris", 385), ("Lyon", 690), ("Marseille", 800)],
    "Marseille": [("Lyon", 315), ("Nantes", 800)]
}

print("Voisins de Paris :", reseau["Paris"])
print("Nombre de sommets :", len(reseau))

### 2.2 Matrice d'adjacence

On utilise un tableau 2D de taille $n \times n$. La valeur `matrice[i][j]` represente le poids de l'arete entre les sommets $i$ et $j$, ou 0 s'il n'y a pas d'arete. Pour un graphe non oriente, la matrice est symetrique.

In [ ]:
# Correspondance sommets <-> indices
villes = ["Paris", "Lyon", "Lille", "Nantes", "Marseille"]
indice = {ville: i for i, ville in enumerate(villes)}

n = len(villes)
matrice = [[0] * n for _ in range(n)]

aretes = [
    ("Paris", "Lyon", 460),
    ("Paris", "Lille", 220),
    ("Paris", "Nantes", 385),
    ("Lyon", "Marseille", 315),
    ("Lyon", "Nantes", 690),
    ("Nantes", "Marseille", 800),
]

for ville_a, ville_b, poids in aretes:
    i, j = indice[ville_a], indice[ville_b]
    matrice[i][j] = poids
    matrice[j][i] = poids  # Symetrie : graphe non oriente

# Affichage
print(f"{'':>12}", end="")
for ville in villes:
    print(f"{ville:>12}", end="")
print()

for i, ville in enumerate(villes):
    print(f"{ville:>12}", end="")
    for j in range(n):
        print(f"{matrice[i][j]:>12}", end="")
    print()

### 2.3 Comparaison des representations

| Critere | Liste d'adjacence | Matrice d'adjacence |
|---|---|---|
| Memoire | $O(S + A)$ | $O(S^2)$ |
| Tester si une arete existe | $O(\text{degre})$ | $O(1)$ |
| Lister les voisins | $O(\text{degre})$ | $O(S)$ |
| Ajout d'une arete | $O(1)$ | $O(1)$ |

Ou $S$ est le nombre de sommets et $A$ le nombre d'aretes.

## 3. Proprietes d'un graphe

Avant d'appliquer un algorithme sur un graphe, il est souvent necessaire de verifier certaines proprietes : le degre des sommets, la connexite, l'existence de cycles, etc.

In [ ]:
def calculer_degres(graphe):
    """Retourne un dictionnaire associant chaque sommet a son degre."""
    return {sommet: len(voisins) for sommet, voisins in graphe.items()}


def somme_degres(graphe):
    """Retourne la somme des degres de tous les sommets."""
    return sum(len(voisins) for voisins in graphe.values())


degres = calculer_degres(reseau)
print("Degres :", degres)
print("Somme des degres :", somme_degres(reseau))
print("Nombre d'aretes :", somme_degres(reseau) // 2)
# Lemme des poignees de main : la somme des degres vaut 2 fois le nombre d'aretes

## 4. Parcours de graphe : BFS et DFS

Les deux algorithmes fondamentaux de parcours permettent de visiter tous les sommets accessibles depuis un sommet de depart.

### 4.1 Parcours en largeur (BFS - Breadth-First Search)

Le BFS explore le graphe couche par couche : d'abord les voisins directs, puis les voisins de voisins, etc. Il utilise une **file** (FIFO).

In [ ]:
from collections import deque


def bfs(graphe, depart):
    """
    Parcours en largeur d'un graphe a partir d'un sommet de depart.

    Retourne la liste des sommets dans l'ordre de visite.
    """
    visites = set()
    file = deque([depart])
    ordre_visite = []

    visites.add(depart)

    while file:
        sommet = file.popleft()
        ordre_visite.append(sommet)

        for voisin, _poids in graphe[sommet]:
            if voisin not in visites:
                visites.add(voisin)
                file.append(voisin)

    return ordre_visite


print("BFS depuis Paris :", bfs(reseau, "Paris"))

### 4.2 Parcours en profondeur (DFS - Depth-First Search)

Le DFS explore le graphe en s'enfoncant le plus possible dans une branche avant de revenir en arriere (backtracking). Il utilise une **pile** (LIFO).

In [ ]:
def dfs(graphe, depart):
    """
    Parcours en profondeur d'un graphe a partir d'un sommet de depart.

    Retourne la liste des sommets dans l'ordre de visite.
    """
    visites = set()
    pile = [depart]
    ordre_visite = []

    while pile:
        sommet = pile.pop()

        if sommet not in visites:
            visites.add(sommet)
            ordre_visite.append(sommet)

            for voisin, _poids in graphe[sommet]:
                if voisin not in visites:
                    pile.append(voisin)

    return ordre_visite


print("DFS depuis Paris :", dfs(reseau, "Paris"))

### 4.3 Test de connexite

Un graphe est connexe si et seulement si un parcours (BFS ou DFS) depuis n'importe quel sommet visite tous les sommets.

In [ ]:
def est_connexe(graphe):
    """Verifie si un graphe non oriente est connexe."""
    if not graphe:
        return True

    depart = next(iter(graphe))
    sommets_atteints = bfs(graphe, depart)

    return len(sommets_atteints) == len(graphe)


print("Le reseau est connexe :", est_connexe(reseau))

# Ajoutons un sommet isole pour tester
reseau_non_connexe = dict(reseau)
reseau_non_connexe["Strasbourg"] = []
print("Avec Strasbourg isole :", est_connexe(reseau_non_connexe))

## 5. Plus court chemin : algorithme de Dijkstra

L'algorithme de Dijkstra permet de trouver le plus court chemin entre un sommet source et tous les autres sommets d'un graphe pondere a poids positifs.

Le principe est le suivant :
1. Initialiser la distance de la source a 0 et toutes les autres a l'infini
2. A chaque etape, selectionner le sommet non visite le plus proche
3. Mettre a jour les distances de ses voisins si un chemin plus court est trouve
4. Repeter jusqu'a avoir visite tous les sommets

In [ ]:
import heapq


def dijkstra(graphe, source):
    """
    Algorithme de Dijkstra pour le plus court chemin.

    Retourne deux dictionnaires :
    - distances : distance minimale de la source a chaque sommet
    - predecesseurs : sommet precedent sur le plus court chemin
    """
    distances = {sommet: float("inf") for sommet in graphe}
    predecesseurs = {sommet: None for sommet in graphe}
    distances[source] = 0

    # File de priorite : (distance, sommet)
    file_priorite = [(0, source)]

    while file_priorite:
        distance_courante, sommet_courant = heapq.heappop(file_priorite)

        # Si on a deja trouve un chemin plus court, on ignore
        if distance_courante > distances[sommet_courant]:
            continue

        for voisin, poids in graphe[sommet_courant]:
            nouvelle_distance = distance_courante + poids

            if nouvelle_distance < distances[voisin]:
                distances[voisin] = nouvelle_distance
                predecesseurs[voisin] = sommet_courant
                heapq.heappush(file_priorite, (nouvelle_distance, voisin))

    return distances, predecesseurs


distances_depuis_paris, predecesseurs = dijkstra(reseau, "Paris")

print("Distances depuis Paris :")
for ville, dist in sorted(distances_depuis_paris.items(), key=lambda x: x[1]):
    print(f"  {ville} : {dist} km")

In [ ]:
def reconstruire_chemin(predecesseurs, destination):
    """Reconstruit le chemin depuis la source jusqu'a la destination."""
    chemin = []
    sommet = destination

    while sommet is not None:
        chemin.append(sommet)
        sommet = predecesseurs[sommet]

    chemin.reverse()
    return chemin


chemin_marseille = reconstruire_chemin(predecesseurs, "Marseille")
print("Plus court chemin Paris -> Marseille :", " -> ".join(chemin_marseille))
print("Distance totale :", distances_depuis_paris["Marseille"], "km")

## 6. Graphes orientes et applications

Dans un graphe oriente, les arcs ont un sens. Cela permet de modeliser des situations ou les trajets ne sont pas symetriques (sens uniques, couts differents selon la direction, etc.).

In [ ]:
# Graphe oriente : reseau de livraison avec sens uniques
reseau_oriente = {
    "Entrepot":  [("Client_A", 30), ("Client_B", 50)],
    "Client_A":  [("Client_C", 20)],
    "Client_B":  [("Client_A", 15), ("Client_C", 40)],
    "Client_C":  [("Entrepot", 35)],
}


def degre_entrant(graphe, sommet):
    """Calcule le degre entrant d'un sommet dans un graphe oriente."""
    compteur = 0
    for voisins in graphe.values():
        for voisin, _poids in voisins:
            if voisin == sommet:
                compteur += 1
    return compteur


def degre_sortant(graphe, sommet):
    """Calcule le degre sortant d'un sommet dans un graphe oriente."""
    return len(graphe.get(sommet, []))


for sommet in reseau_oriente:
    d_in = degre_entrant(reseau_oriente, sommet)
    d_out = degre_sortant(reseau_oriente, sommet)
    print(f"{sommet} : degre entrant = {d_in}, degre sortant = {d_out}")

## 7. Matrice des distances et graphe complet

Pour le projet ADEME (probleme du voyageur de commerce), on travaille souvent avec un graphe complet ou chaque paire de villes est reliee. On peut generer automatiquement la matrice des distances a partir de coordonnees.

In [ ]:
import math
import random

random.seed(42)


def generer_villes(nombre, x_max=100, y_max=100):
    """Genere des villes avec des coordonnees aleatoires."""
    return {f"V{i}": (random.randint(0, x_max), random.randint(0, y_max)) for i in range(nombre)}


def construire_matrice_distances(villes_coords):
    """Construit la matrice des distances euclidiennes entre toutes les paires de villes."""
    noms = list(villes_coords.keys())
    taille = len(noms)
    matrice = [[0.0] * taille for _ in range(taille)]

    for i in range(taille):
        xi, yi = villes_coords[noms[i]]
        for j in range(i + 1, taille):
            xj, yj = villes_coords[noms[j]]
            dist = math.sqrt((xj - xi) ** 2 + (yj - yi) ** 2)
            matrice[i][j] = round(dist, 2)
            matrice[j][i] = round(dist, 2)

    return noms, matrice


mes_villes = generer_villes(6)
print("Coordonnees :")
for nom, coords in mes_villes.items():
    print(f"  {nom} : {coords}")

noms_villes, mat_dist = construire_matrice_distances(mes_villes)

print("\nMatrice des distances :")
print(f"{'':>6}", end="")
for nom in noms_villes:
    print(f"{nom:>8}", end="")
print()
for i, nom in enumerate(noms_villes):
    print(f"{nom:>6}", end="")
    for j in range(len(noms_villes)):
        print(f"{mat_dist[i][j]:>8}", end="")
    print()

## 8. Introduction au probleme du voyageur de commerce (TSP)

Le probleme du voyageur de commerce consiste a trouver la tournee la plus courte passant par toutes les villes exactement une fois et revenant au point de depart. C'est un probleme central du projet ADEME.

Voici une approche gloutonne simple : a chaque etape, on visite la ville non visitee la plus proche.

In [ ]:
def tsp_glouton(matrice_dist, depart=0):
    """
    Heuristique gloutonne pour le TSP : plus proche voisin.

    Retourne la tournee et sa distance totale.
    """
    n = len(matrice_dist)
    visites = {depart}
    tournee = [depart]
    courant = depart
    distance_totale = 0.0

    while len(visites) < n:
        # Trouver la ville non visitee la plus proche
        plus_proche = None
        dist_min = float("inf")

        for ville in range(n):
            if ville not in visites and matrice_dist[courant][ville] < dist_min:
                plus_proche = ville
                dist_min = matrice_dist[courant][ville]

        visites.add(plus_proche)
        tournee.append(plus_proche)
        distance_totale += dist_min
        courant = plus_proche

    # Retour au depart
    distance_totale += matrice_dist[courant][depart]
    tournee.append(depart)

    return tournee, round(distance_totale, 2)


tournee, distance = tsp_glouton(mat_dist)
noms_tournee = [noms_villes[i] for i in tournee]

print("Tournee gloutonne :", " -> ".join(noms_tournee))
print("Distance totale :", distance)

## 9. Exercices de synthese

Ces exercices permettent de verifier la maitrise des notions vues dans cette sequence.

### Exercice 1 : Detecter un cycle

Implementez une fonction `possede_cycle(graphe)` qui determine si un graphe non oriente contient au moins un cycle. Indice : lors d'un DFS, si on rencontre un sommet deja visite qui n'est pas le parent du sommet courant, il y a un cycle.

In [ ]:
def possede_cycle(graphe):
    """
    Detecte la presence d'un cycle dans un graphe non oriente.

    A COMPLETER par les etudiants.
    """
    pass


# Test : le reseau contient des cycles (Paris-Lyon-Nantes-Paris par exemple)
# print(possede_cycle(reseau))

### Exercice 2 : Dijkstra sur matrice

Adaptez l'algorithme de Dijkstra pour qu'il fonctionne sur une matrice d'adjacence au lieu d'une liste d'adjacence. Testez-le sur la matrice de distances generee a la section 7.

In [ ]:
def dijkstra_matrice(matrice_adj, source):
    """
    Dijkstra sur une matrice d'adjacence.

    A COMPLETER par les etudiants.
    """
    pass


# Test
# distances = dijkstra_matrice(mat_dist, 0)
# print(distances)

### Exercice 3 : Comparer les heuristiques

L'heuristique gloutonne ne donne pas toujours la meilleure solution. Implementez une fonction qui teste l'heuristique du plus proche voisin en partant de chaque ville, et retourne la meilleure tournee trouvee.

In [ ]:
def tsp_glouton_meilleur_depart(matrice_dist):
    """
    Teste l'heuristique gloutonne depuis chaque sommet et retourne la meilleure tournee.

    A COMPLETER par les etudiants.
    """
    pass


# Test
# meilleure_tournee, meilleure_distance = tsp_glouton_meilleur_depart(mat_dist)
# print("Meilleure tournee :", meilleure_tournee)
# print("Distance :", meilleure_distance)